# End-to-End ETL & Data Analysis Pipeline: Olist Brazilian E-Commerce

**Author:** Bipana Khadka  |  **Database:** SQLite  |  **Tools:** Python, Pandas, Matplotlib, SQLAlchemy, SQL

**Pipeline:** Raw Data → Extract → EDA → Clean → Transform → Analyze → Load → SQL Reports → Insights

This notebook runs the whole pipeline from top to bottom. The same code also lives in `run_pipeline.py`, so anyone can rebuild the database with one command:

```
python run_pipeline.py
```

**Dataset:** [Brazilian E-Commerce Public Dataset by Olist (Kaggle)](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce): 9 CSV files with orders placed on a Brazilian online marketplace (2016 to 2018). Put the 9 files in `data/raw/` before running. Money values are in Brazilian reais (R$).

## 0. Setup

Imports, folder paths and a few helpers. Charts use one calm blue and a light grid so the numbers stay the focus.

In [ ]:
import re
import sqlite3
import textwrap
import unicodedata
import warnings
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd

# In the notebook, charts are shown. In `python run_pipeline.py`, they are only saved to files.
try:
    get_ipython()
except NameError:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from sqlalchemy import create_engine, event, text

warnings.filterwarnings("ignore")
try:
    display
except NameError:          # running as a plain script
    display = print

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

# ---- Paths (work from the notebook folder or from the project root) ----
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
FIG_DIR = ROOT / "docs" / "figures"
OUT_DIR = ROOT / "outputs" / "sql_reports"
DB_PATH = ROOT / "database" / "olist.db"
SQL_DIR = ROOT / "sql"
for folder in (PROCESSED_DIR, FIG_DIR, OUT_DIR, DB_PATH.parent):
    folder.mkdir(parents=True, exist_ok=True)

# ---- Things we collect while running, used for the final report ----
STATS = {}        # numbers for the report
CLEAN_LOG = []    # (table, what we did, rows affected, why)
INSIGHTS = []     # business insights written from the analysis


def log(table, action, n, why):
    """Record one cleaning step so it can be explained in the report."""
    CLEAN_LOG.append((table, action, int(n), why))
    print(f"[{table}] {action}: {int(n):,}")


# ---- Chart style ----
BLUE, INK, MUTED, GRID = "#2a78d6", "#0b0b0b", "#52514e", "#e3e2de"
THOUSANDS = FuncFormatter(lambda v, _: f"{v:,.0f}")     # 80000 -> 80,000
plt.rcParams.update({
    "figure.figsize": (9, 4.5), "figure.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GRID, "axes.labelcolor": MUTED,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "font.size": 10,
})


def save_fig(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / name, dpi=130)
    plt.show()
    plt.close()


def strip_accents(text):
    return unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode("ascii")


def clean_text(series):
    """trim spaces, lower-case, remove accents, then Title Case (e.g. ' São PAULO ' -> 'Sao Paulo')."""
    s = series.astype("string").str.strip().str.lower()
    mapping = {u: strip_accents(u) for u in s.dropna().unique()}   # map unique values only: fast
    return s.map(mapping).str.title()


print("Project folder:", ROOT)

## 1. Data Acquisition & Extraction

All 9 CSV files are read with Pandas into a dictionary called `raw`.

**The raw data is never changed.** The files stay untouched in `data/raw/`, and every cleaning step below works on a copy of `raw`.

In [ ]:
FILES = {
    "orders":               "olist_orders_dataset.csv",
    "customers":            "olist_customers_dataset.csv",
    "order_items":          "olist_order_items_dataset.csv",
    "payments":             "olist_order_payments_dataset.csv",
    "reviews":              "olist_order_reviews_dataset.csv",
    "products":             "olist_products_dataset.csv",
    "sellers":              "olist_sellers_dataset.csv",
    "geolocation":          "olist_geolocation_dataset.csv",
    "category_translation":  "product_category_name_translation.csv",
}

missing_files = [f for f in FILES.values() if not (RAW_DIR / f).exists()]
if missing_files:
    raise FileNotFoundError(f"Put these files in {RAW_DIR}:\n  " + "\n  ".join(missing_files))

raw = {name: pd.read_csv(RAW_DIR / filename) for name, filename in FILES.items()}   # EXTRACT
print(f"Extracted {len(raw)} files from {RAW_DIR}")

## 2. Exploratory Data Analysis (EDA)

EDA is done on the **raw** data, before any cleaning, so we can see what is wrong with it.

### 2.1 Basic exploration: size, columns, data types, samples

In [ ]:
overview = pd.DataFrame({
    "rows":    {k: v.shape[0] for k, v in raw.items()},
    "columns": {k: v.shape[1] for k, v in raw.items()},
    "memory_mb": {k: round(v.memory_usage(deep=True).sum() / 1e6, 1) for k, v in raw.items()},
})
display(overview)
total_cols = int(overview["columns"].sum())
print(f"\n{len(raw)} tables, {total_cols} columns in total, {overview['rows'].sum():,} rows in total")
STATS["raw_shapes"] = {k: (int(v.shape[0]), int(v.shape[1])) for k, v in raw.items()}
STATS["total_columns"] = total_cols

In [ ]:
# Column names and data types of every table
for name, d in raw.items():
    cols = ", ".join(f"{c} ({str(t)})" for c, t in d.dtypes.items())
    print(f"\n{name}  {d.shape}\n  {cols}")

In [ ]:
# Sample records
for name in ["orders", "order_items", "customers", "products", "payments", "reviews"]:
    print(f"\n--- sample records: {name}")
    display(raw[name].head(3))

In [ ]:
# Summary statistics of the numeric columns
display(raw["order_items"][["price", "freight_value"]].describe().round(2))
display(raw["payments"][["payment_installments", "payment_value"]].describe().round(2))
display(raw["products"].describe().round(1))
display(raw["reviews"][["review_score"]].describe().round(2))

In [ ]:
# Number of unique values in the important columns
unique_counts = pd.Series({
    "orders.order_id":                      raw["orders"]["order_id"].nunique(),
    "orders.order_status":                  raw["orders"]["order_status"].nunique(),
    "customers.customer_id":                raw["customers"]["customer_id"].nunique(),
    "customers.customer_unique_id":         raw["customers"]["customer_unique_id"].nunique(),
    "customers.customer_state":             raw["customers"]["customer_state"].nunique(),
    "customers.customer_city":              raw["customers"]["customer_city"].nunique(),
    "products.product_id":                  raw["products"]["product_id"].nunique(),
    "products.product_category_name":       raw["products"]["product_category_name"].nunique(),
    "sellers.seller_id":                    raw["sellers"]["seller_id"].nunique(),
    "order_items.product_id":               raw["order_items"]["product_id"].nunique(),
    "payments.payment_type":                raw["payments"]["payment_type"].nunique(),
    "reviews.review_score":                 raw["reviews"]["review_score"].nunique(),
}, name="unique_values")
display(unique_counts.to_frame())
STATS["unique_counts"] = unique_counts.astype(int).to_dict()

**What this tells us:** `customer_id` is created for every order, while `customer_unique_id` is the real person, so a customer who orders twice gets two `customer_id` values. Customer analysis must use `customer_unique_id`.

### 2.2 Data quality analysis

We check: missing values, duplicates, wrong data types, invalid values, outliers, unexpected categories and inconsistent formatting.

In [ ]:
# ---- Missing values ----
rows = []
for name, d in raw.items():
    nulls = d.isna().sum()
    for col, count in nulls[nulls > 0].items():
        rows.append((name, col, int(count), round(100 * count / len(d), 2)))
null_report = pd.DataFrame(rows, columns=["table", "column", "missing", "missing_pct"]) \
                .sort_values("missing", ascending=False).reset_index(drop=True)
display(null_report)
STATS["null_report"] = null_report

In [ ]:
# ---- Duplicates ----
full_row_dups = {name: int(d.duplicated().sum()) for name, d in raw.items()}
key_dups = {
    "orders: repeated order_id":                         int(raw["orders"]["order_id"].duplicated().sum()),
    "customers: repeated customer_id":                   int(raw["customers"]["customer_id"].duplicated().sum()),
    "products: repeated product_id":                     int(raw["products"]["product_id"].duplicated().sum()),
    "sellers: repeated seller_id":                       int(raw["sellers"]["seller_id"].duplicated().sum()),
    "order_items: repeated (order_id, order_item_id)":    int(raw["order_items"].duplicated(["order_id", "order_item_id"]).sum()),
    "reviews: repeated review_id":                       int(raw["reviews"]["review_id"].duplicated().sum()),
    "reviews: repeated (review_id, order_id)":           int(raw["reviews"].duplicated(["review_id", "order_id"]).sum()),
}
print("Fully duplicated rows per table:")
display(pd.Series(full_row_dups, name="duplicate_rows").to_frame())
print("Duplicated keys:")
display(pd.Series(key_dups, name="duplicates").to_frame())
STATS["full_row_dups"], STATS["key_dups"] = full_row_dups, key_dups

In [ ]:
# ---- Incorrect data types: date columns are read as plain text ----
DATE_COLS = {
    "orders": ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
               "order_delivered_customer_date", "order_estimated_delivery_date"],
    "order_items": ["shipping_limit_date"],
    "reviews": ["review_creation_date", "review_answer_timestamp"],
}
wrong_types = pd.DataFrame(
    [(t, c, str(raw[t][c].dtype)) for t, cols in DATE_COLS.items() for c in cols],
    columns=["table", "column", "current_dtype"])
wrong_types["should_be"] = "datetime"
display(wrong_types)
STATS["wrong_types"] = len(wrong_types)

In [ ]:
# ---- Invalid or inconsistent values ----
o = raw["orders"]
purchase = pd.to_datetime(o["order_purchase_timestamp"], errors="coerce")
delivered = pd.to_datetime(o["order_delivered_customer_date"], errors="coerce")

invalid = {
    "payments: payment_value = 0":                      int((raw["payments"]["payment_value"] == 0).sum()),
    "payments: payment_type = 'not_defined'":           int((raw["payments"]["payment_type"] == "not_defined").sum()),
    "payments: payment_installments = 0":               int((raw["payments"]["payment_installments"] == 0).sum()),
    "order_items: price <= 0":                          int((raw["order_items"]["price"] <= 0).sum()),
    "order_items: freight_value < 0":                   int((raw["order_items"]["freight_value"] < 0).sum()),
    "products: weight_g = 0":                           int((raw["products"]["product_weight_g"] == 0).sum()),
    "products: missing category name":                  int(raw["products"]["product_category_name"].isna().sum()),
    "reviews: review_score outside 1 to 5":             int((~raw["reviews"]["review_score"].between(1, 5)).sum()),
    "orders: 'delivered' but no delivery date":         int(((o["order_status"] == "delivered") & delivered.isna()).sum()),
    "orders: delivered before purchase":                int((delivered < purchase).sum()),
    "orders: not delivered but has a delivery date":    int(((o["order_status"] != "delivered") & delivered.notna()).sum()),
}
invalid_df = pd.Series(invalid, name="rows").to_frame()
display(invalid_df)
STATS["invalid"] = invalid

In [ ]:
# ---- Outliers (IQR rule: beyond Q3 + 1.5 x IQR) ----
def iqr_outliers(series):
    s = series.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    upper = q3 + 1.5 * (q3 - q1)
    return int((s > upper).sum()), round(float(upper), 2), round(float(s.max()), 2)

delivery_days_raw = (delivered - purchase).dt.total_seconds() / 86400
outlier_rows = []
for label, series in [("order_items.price", raw["order_items"]["price"]),
                      ("order_items.freight_value", raw["order_items"]["freight_value"]),
                      ("payments.payment_value", raw["payments"]["payment_value"]),
                      ("delivery time (days)", delivery_days_raw)]:
    n_out, upper, biggest = iqr_outliers(series)
    outlier_rows.append((label, n_out, upper, biggest))
outliers = pd.DataFrame(outlier_rows, columns=["column", "outliers", "upper_limit", "max_value"])
display(outliers)
STATS["outliers"] = outliers

In [ ]:
# ---- Unexpected categories and inconsistent formatting ----
print("Order status:");   display(raw["orders"]["order_status"].value_counts().to_frame())
print("Payment type:");   display(raw["payments"]["payment_type"].value_counts().to_frame())

translated = set(raw["category_translation"]["product_category_name"])
untranslated = sorted(set(raw["products"]["product_category_name"].dropna()) - translated)
print(f"Product categories with no English translation: {len(untranslated)} -> {untranslated}")

def formatting_problems(series):
    s = series.dropna().astype(str)
    return {"extra spaces": int((s != s.str.strip()).sum()),
            "not lower-case": int((s != s.str.lower()).sum()),
            "has accents": int(s.str.contains(r"[^\x00-\x7F]").sum())}

formatting = pd.DataFrame({
    "customers.city": formatting_problems(raw["customers"]["customer_city"]),
    "sellers.city": formatting_problems(raw["sellers"]["seller_city"]),
    "geolocation.city": formatting_problems(raw["geolocation"]["geolocation_city"]),
}).T
display(formatting)
STATS["untranslated"] = untranslated
STATS["formatting"] = formatting

### 2.3 Exploratory analysis (on raw data)

First look at the business, using `groupby()`, `agg()`, `value_counts()`, `sort_values()`, `loc[]`, `iloc[]`, filtering and calculated columns.

In [ ]:
items_raw = raw["order_items"]

# groupby + agg + sort_values: best-selling products by revenue
by_product = (items_raw.groupby("product_id")
              .agg(units_sold=("order_id", "count"), revenue=("price", "sum"))
              .sort_values("revenue", ascending=False))
print("Top 5 products by revenue (head / iloc):")
display(by_product.iloc[:5].round(2))

# loc[] + filtering: expensive items
expensive = items_raw.loc[items_raw["price"] > 1000, ["order_id", "product_id", "price"]]
print(f"Items priced above R$ 1,000: {len(expensive):,} ({100 * len(expensive) / len(items_raw):.2f}% of all items)")

# calculated column: total value of an item line = price + freight
items_calc = items_raw.assign(total_price=items_raw["price"] + items_raw["freight_value"])
print("Average item total (price + freight): R$", round(items_calc["total_price"].mean(), 2))

# value_counts: customers by state
print("\nCustomers by state (top 5):")
display(raw["customers"]["customer_state"].value_counts().head(5).to_frame())

In [ ]:
# Chart 1: orders per month (trend over time)
purchase_ts_raw = pd.to_datetime(raw["orders"]["order_purchase_timestamp"], errors="coerce")
orders_per_month = purchase_ts_raw.dt.to_period("M").value_counts().sort_index()

fig, ax = plt.subplots()
ax.plot(orders_per_month.index.to_timestamp(), orders_per_month.values, color=BLUE, linewidth=2)
ax.set_title("Orders per month")
ax.set_ylabel("Number of orders")
peak = orders_per_month.idxmax()
ax.annotate(f"Peak: {peak}  ({orders_per_month.max():,} orders)",
            (peak.to_timestamp(), orders_per_month.max()), xytext=(8, -4),
            textcoords="offset points", color=INK, fontsize=9)
save_fig("01_orders_per_month.png")
STATS["eda_peak_orders_month"] = (str(peak), int(orders_per_month.max()))

In [ ]:
# Chart 2: review scores
score_counts = raw["reviews"]["review_score"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(score_counts.index.astype(str), score_counts.values, color=BLUE, width=0.6)
ax.set_title("Review score distribution")
ax.set_xlabel("Review score (1 = worst, 5 = best)")
ax.set_ylabel("Number of reviews")
ax.grid(axis="x", visible=False)
for x, v in zip(score_counts.index.astype(str), score_counts.values):
    ax.text(x, v, f"{v:,}", ha="center", va="bottom", fontsize=9, color=INK)
save_fig("02_review_scores.png")
STATS["eda_five_star_pct"] = round(100 * (raw["reviews"]["review_score"] == 5).mean(), 1)

# Chart 3: item prices (long right tail, so the chart stops at the 99th percentile)
cap = items_raw["price"].quantile(0.99)
fig, ax = plt.subplots()
ax.hist(items_raw["price"].clip(upper=cap), bins=40, color=BLUE, edgecolor="white", linewidth=0.5)
ax.set_title(f"Item price distribution (capped at the 99th percentile, R$ {cap:,.0f})")
ax.set_xlabel("Price (R$)")
ax.set_ylabel("Number of items")
ax.grid(axis="x", visible=False)
save_fig("03_price_distribution.png")
STATS["eda_price_median"] = round(float(items_raw["price"].median()), 2)

**EDA summary:** the data is well organised, but it has real problems we will fix next: dates stored as text, missing delivery dates, duplicated review rows, a huge duplicated geolocation file, missing product categories, categories without an English name, accents and capital letters in city names, a few invalid payment rows, and a long tail of high prices that is real (not an error).

## 3. Data Cleaning & Transformation

Every step below fixes a problem found in the EDA. Each step is written to `CLEAN_LOG` together with the **reason**, so it can be explained in the report.

Order of work: clean each table, then remove rows that point to missing parents (so the database foreign keys will not fail), then build new columns and analysis tables.

### 3.1 Orders: dates, status, duplicates, new delivery columns

In [ ]:
orders = raw["orders"].copy()

# 1) Text -> datetime, and clearer column names
DATE_RENAME = {
    "order_purchase_timestamp":       "purchase_ts",
    "order_approved_at":              "approved_ts",
    "order_delivered_carrier_date":   "delivered_carrier_ts",
    "order_delivered_customer_date":  "delivered_customer_ts",
    "order_estimated_delivery_date":  "estimated_delivery_date",
}
for col in DATE_RENAME:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")
orders = orders.rename(columns=DATE_RENAME)
log("orders", "date columns converted from text to datetime", len(DATE_RENAME),
    "text dates cannot be sorted, grouped by month or subtracted")

# 2) Standardise status text
orders["order_status"] = orders["order_status"].str.strip().str.lower()

# 3) Duplicate orders
n = orders.duplicated("order_id").sum()
orders = orders.drop_duplicates("order_id")
log("orders", "duplicate order_id rows removed", n, "order_id is the primary key")

# 4) An order with no purchase time cannot be analysed
n = orders["purchase_ts"].isna().sum()
orders = orders.dropna(subset=["purchase_ts"])
log("orders", "rows without purchase time removed", n, "purchase time is needed for every time-based report")

# 5) Delivered before it was bought is impossible: blank the bad delivery date
bad = orders["delivered_customer_ts"] < orders["purchase_ts"]
orders.loc[bad, "delivered_customer_ts"] = pd.NaT
log("orders", "impossible delivery dates set to missing", bad.sum(), "delivery cannot happen before purchase")

# 6) New columns for time analysis and delivery performance
orders["purchase_year"] = orders["purchase_ts"].dt.year
orders["purchase_month"] = orders["purchase_ts"].dt.month
orders["delivery_days"] = ((orders["delivered_customer_ts"] - orders["purchase_ts"])
                           .dt.total_seconds() / 86400).round(2)
orders["delay_days"] = (orders["delivered_customer_ts"].dt.normalize()
                        - orders["estimated_delivery_date"].dt.normalize()).dt.days
orders["is_late"] = np.where(orders["delay_days"].isna(), np.nan, (orders["delay_days"] > 0).astype(float))
log("orders", "new columns: purchase_year, purchase_month, delivery_days, delay_days, is_late",
    len(orders), "needed to answer questions about monthly trends and late deliveries")
print(f"\nOrders still without a delivery date (not delivered yet or canceled): {orders['delivered_customer_ts'].isna().sum():,} (kept as missing: this is real information)")

### 3.2 Customers and sellers: text formatting

In [ ]:
customers = raw["customers"].rename(columns={
    "customer_zip_code_prefix": "zip_code_prefix", "customer_city": "city", "customer_state": "state"}).copy()
n = customers.duplicated("customer_id").sum()
customers = customers.drop_duplicates("customer_id")
log("customers", "duplicate customer_id rows removed", n, "customer_id is the primary key")
changed = (customers["city"] != clean_text(customers["city"])).sum()
customers["city"] = clean_text(customers["city"])
customers["state"] = customers["state"].str.strip().str.upper()
log("customers", "city names standardised (spaces, accents, capital letters)", changed,
    "'sao paulo', 'São Paulo' and 'SAO PAULO ' must be one group in reports")

sellers = raw["sellers"].rename(columns={
    "seller_zip_code_prefix": "zip_code_prefix", "seller_city": "city", "seller_state": "state"}).copy()
n = sellers.duplicated("seller_id").sum()
sellers = sellers.drop_duplicates("seller_id")
log("sellers", "duplicate seller_id rows removed", n, "seller_id is the primary key")
changed = (sellers["city"] != clean_text(sellers["city"])).sum()
sellers["city"] = clean_text(sellers["city"])
sellers["state"] = sellers["state"].str.strip().str.upper()
log("sellers", "city names standardised (spaces, accents, capital letters)", changed,
    "same reason as customer cities")

### 3.3 Products and categories: missing values, wrong spellings, translation

In [ ]:
products = raw["products"].rename(columns={
    "product_category_name": "category_name",
    "product_name_lenght": "name_length",              # the original column name has a spelling mistake
    "product_description_lenght": "description_length",
    "product_photos_qty": "photos_qty",
    "product_weight_g": "weight_g",
    "product_length_cm": "length_cm",
    "product_height_cm": "height_cm",
    "product_width_cm": "width_cm",
}).copy()

n = products.duplicated("product_id").sum()
products = products.drop_duplicates("product_id")
log("products", "duplicate product_id rows removed", n, "product_id is the primary key")

# Missing category -> 'unknown' (keeps the product and its sales in the reports)
products["category_name"] = products["category_name"].str.strip().str.lower()
n = products["category_name"].isna().sum()
products["category_name"] = products["category_name"].fillna("unknown")
log("products", "missing category set to 'unknown'", n, "removing them would delete real sales")

# Weight of 0 g is impossible -> treat as missing
n = (products["weight_g"] == 0).sum()
products.loc[products["weight_g"] == 0, "weight_g"] = np.nan
log("products", "weight of 0 g treated as missing", n, "a product cannot weigh 0 g")

# Fill missing numeric details with the median (the middle value is not affected by extremes)
for col in ["name_length", "description_length", "photos_qty", "weight_g", "length_cm", "height_cm", "width_cm"]:
    n = products[col].isna().sum()
    if n:
        products[col] = products[col].fillna(products[col].median())
        log("products", f"missing {col} filled with the median", n, "keeps the row; median is safe against outliers")
for col in ["name_length", "description_length", "photos_qty"]:
    products[col] = products[col].round().astype(int)

# Category table: Portuguese name -> English name
categories = raw["category_translation"].rename(columns={"product_category_name": "category_name",
                                                         "product_category_name_english": "category_name_en"}).copy()
categories["category_name"] = categories["category_name"].str.strip().str.lower()
categories["category_name_en"] = categories["category_name_en"].str.strip().str.lower()

used = set(products["category_name"].unique()) | {"unknown"}
extra = sorted(used - set(categories["category_name"]))
extra_df = pd.DataFrame({"category_name": extra,
                         "category_name_en": [c.replace("_", " ") for c in extra]})
categories = pd.concat([categories, extra_df], ignore_index=True).drop_duplicates("category_name")
log("product_categories", "categories without an English name added (underscores -> spaces)", len(extra),
    f"these categories had no translation: {extra}")

### 3.4 Order items, payments and reviews

In [ ]:
items = raw["order_items"].rename(columns={"shipping_limit_date": "shipping_limit_ts"}).copy()
items["shipping_limit_ts"] = pd.to_datetime(items["shipping_limit_ts"], errors="coerce")
log("order_items", "shipping_limit_ts converted to datetime", len(items), "text dates cannot be used in calculations")

n = items.duplicated(["order_id", "order_item_id"]).sum()
items = items.drop_duplicates(["order_id", "order_item_id"])
log("order_items", "duplicate (order_id, order_item_id) rows removed", n, "this pair is the primary key")

n = ((items["price"] < 0) | (items["freight_value"] < 0)).sum()
items = items[(items["price"] >= 0) & (items["freight_value"] >= 0)]
log("order_items", "rows with negative price or freight removed", n, "money values cannot be negative")

# Outliers: very expensive items are REAL purchases, so we keep them
q1, q3 = items["price"].quantile([0.25, 0.75])
n = (items["price"] > q3 + 1.5 * (q3 - q1)).sum()
log("order_items", "price outliers kept (checked, not removed)", n,
    "high prices are genuine purchases (for example furniture, appliances), not errors")

items["total_price"] = (items["price"] + items["freight_value"]).round(2)
log("order_items", "new column total_price = price + freight_value", len(items), "needed for order value reports")

# ---------------- payments ----------------
payments = raw["payments"].copy()
payments["payment_type"] = payments["payment_type"].str.strip().str.lower()
n = (payments["payment_type"] == "not_defined").sum()
payments = payments[payments["payment_type"] != "not_defined"]
log("payments", "rows with payment_type 'not_defined' removed", n, "the payment method is unknown, so the row carries no information")
n = (payments["payment_installments"] == 0).sum()
payments.loc[payments["payment_installments"] == 0, "payment_installments"] = 1
log("payments", "installments of 0 changed to 1", n, "a payment has at least 1 installment")
n = payments.duplicated(["order_id", "payment_sequential"]).sum()
payments = payments.drop_duplicates(["order_id", "payment_sequential"])
log("payments", "duplicate (order_id, payment_sequential) rows removed", n, "this pair is the primary key")

# ---------------- reviews ----------------
reviews = raw["reviews"].rename(columns={
    "review_comment_title": "comment_title", "review_comment_message": "comment_message",
    "review_creation_date": "creation_date", "review_answer_timestamp": "answer_ts"}).copy()
reviews["creation_date"] = pd.to_datetime(reviews["creation_date"], errors="coerce")
reviews["answer_ts"] = pd.to_datetime(reviews["answer_ts"], errors="coerce")
log("reviews", "date columns converted to datetime", 2, "text dates cannot be used in calculations")

n = (~reviews["review_score"].between(1, 5)).sum()
reviews = reviews[reviews["review_score"].between(1, 5)]
log("reviews", "rows with review_score outside 1 to 5 removed", n, "scores must be 1 to 5")

n = reviews.duplicated(["review_id", "order_id"]).sum()
reviews = reviews.sort_values("answer_ts").drop_duplicates(["review_id", "order_id"], keep="last")
log("reviews", "duplicate (review_id, order_id) rows removed (latest answer kept)", n, "the same review was stored more than once")
# Empty comments are normal (most people only give a score), so they stay NULL.

### 3.5 Geolocation: from 1 million rows to one row per zip code

The raw file lists many coordinates for the same zip-code prefix. We remove duplicates and points outside Brazil, then keep one averaged point per zip prefix.

In [ ]:
geo = raw["geolocation"].rename(columns={
    "geolocation_zip_code_prefix": "zip_code_prefix", "geolocation_lat": "latitude",
    "geolocation_lng": "longitude", "geolocation_city": "city", "geolocation_state": "state"}).copy()
rows_before = len(geo)

n = geo.duplicated().sum()
geo = geo.drop_duplicates()
log("geolocation", "fully duplicated rows removed", n, "identical rows add nothing")

inside_brazil = geo["latitude"].between(-34, 6) & geo["longitude"].between(-74, -34)
log("geolocation", "points outside Brazil removed", (~inside_brazil).sum(), "wrong coordinates are outliers")
geo = geo[inside_brazil]

geo["city"] = clean_text(geo["city"])
geo["state"] = geo["state"].str.strip().str.upper()

def most_common(frame, column):
    counts = frame.groupby(["zip_code_prefix", column]).size().reset_index(name="n")
    return (counts.sort_values("n").drop_duplicates("zip_code_prefix", keep="last")
            .set_index("zip_code_prefix")[column])

geolocation = (geo.groupby("zip_code_prefix")
               .agg(latitude=("latitude", "mean"), longitude=("longitude", "mean")).round(6))
geolocation["city"] = most_common(geo, "city")
geolocation["state"] = most_common(geo, "state")
geolocation = geolocation.reset_index()
log("geolocation", f"{rows_before:,} raw rows reduced to one row per zip prefix", len(geolocation),
    "one averaged coordinate per zip code is enough for maps and joins")

### 3.6 Referential integrity: no orphan rows

The database will enforce foreign keys, so every child row must point to a parent that exists.

In [ ]:
def keep_valid(frame, column, valid_ids, table, parent):
    ok = frame[column].isin(valid_ids)
    log(table, f"orphan rows removed ({column} not found in {parent})", (~ok).sum(),
        "a foreign key cannot point to a missing parent")
    return frame[ok]

orders = keep_valid(orders, "customer_id", customers["customer_id"], "orders", "customers")
items = keep_valid(items, "order_id", orders["order_id"], "order_items", "orders")
items = keep_valid(items, "product_id", products["product_id"], "order_items", "products")
items = keep_valid(items, "seller_id", sellers["seller_id"], "order_items", "sellers")
payments = keep_valid(payments, "order_id", orders["order_id"], "payments", "orders")
reviews = keep_valid(reviews, "order_id", orders["order_id"], "reviews", "orders")

STATS["clean_shapes"] = {"orders": len(orders), "customers": len(customers), "sellers": len(sellers),
                         "products": len(products), "product_categories": len(categories),
                         "order_items": len(items), "order_payments": len(payments),
                         "order_reviews": len(reviews), "geolocation": len(geolocation)}
display(pd.Series(STATS["clean_shapes"], name="rows after cleaning").to_frame())

### 3.7 Quality checks (the pipeline stops if one fails)

These are the tests of the clean data. If any check fails, loading is blocked, so a bad file can never reach the database.

In [ ]:
checks = {
    "orders: order_id is unique":                      orders["order_id"].is_unique,
    "customers: customer_id is unique":                customers["customer_id"].is_unique,
    "products: product_id is unique":                  products["product_id"].is_unique,
    "sellers: seller_id is unique":                    sellers["seller_id"].is_unique,
    "order_items: (order_id, order_item_id) is unique": not items.duplicated(["order_id", "order_item_id"]).any(),
    "order_reviews: (review_id, order_id) is unique":  not reviews.duplicated(["review_id", "order_id"]).any(),
    "no missing purchase time":                        orders["purchase_ts"].notna().all(),
    "prices are not negative":                         (items["price"] >= 0).all(),
    "review scores are 1 to 5":                        reviews["review_score"].between(1, 5).all(),
    "every item has a product category":               products["category_name"].notna().all(),
    "every product category has an English name":      products["category_name"].isin(categories["category_name"]).all(),
    "every order has a customer":                      orders["customer_id"].isin(customers["customer_id"]).all(),
    "tables are not empty":                            all(len(t) > 0 for t in (orders, customers, items, products, sellers, payments, reviews)),
}
result = pd.Series(checks, name="passed").to_frame()
display(result)
failed = [name for name, ok in checks.items() if not ok]
assert not failed, f"Data quality checks failed: {failed}"
STATS["quality_checks"] = len(checks)
print(f"\nAll {len(checks)} quality checks passed.")

### 3.8 Transformation: tables ready for analysis

New tables are built for the questions in section 4:

| New table | What it is | Why |
|---|---|---|
| `sales` | one row per sold item, joined with order, category and customer | revenue questions need all of these together |
| `order_totals` | one row per order with its total value | average order value |
| `customer_orders` | one row per real customer, with a segment (One-time / Two orders / 3+ orders) | customer segments |

Canceled and unavailable orders are excluded from revenue.

In [ ]:
sales = (items
         .merge(orders[["order_id", "customer_id", "order_status", "purchase_ts"]], on="order_id")
         .merge(products[["product_id", "category_name"]], on="product_id")
         .merge(categories, on="category_name", how="left")
         .merge(customers[["customer_id", "customer_unique_id", "state"]], on="customer_id"))
sales = sales[~sales["order_status"].isin(["canceled", "unavailable"])].copy()
sales["year_month"] = sales["purchase_ts"].dt.to_period("M").astype(str)
sales["price_band"] = pd.cut(sales["price"], [0, 50, 100, 200, np.inf],
                             labels=["under 50", "50 to 100", "100 to 200", "over 200"], right=False)

order_totals = (sales.groupby("order_id")
                .agg(order_total=("total_price", "sum"), items=("order_item_id", "count"),
                     customer_unique_id=("customer_unique_id", "first"), state=("state", "first"))
                .reset_index())

customer_orders = (order_totals.groupby("customer_unique_id")
                   .agg(orders=("order_id", "nunique"), total_spent=("order_total", "sum")).reset_index())
customer_orders["segment"] = pd.cut(customer_orders["orders"], [0, 1, 2, np.inf],
                                    labels=["One-time", "Two orders", "3+ orders"])

print(f"sales: {len(sales):,} rows | order_totals: {len(order_totals):,} rows | customer_orders: {len(customer_orders):,} rows")
display(sales.head(3))

# Save the cleaned tables (processed layer)
for name, frame in {"orders": orders, "customers": customers, "sellers": sellers, "products": products,
                    "product_categories": categories, "order_items": items, "order_payments": payments,
                    "order_reviews": reviews, "geolocation": geolocation}.items():
    frame.to_csv(PROCESSED_DIR / f"{name}.csv", index=False)
print("Cleaned tables saved in", PROCESSED_DIR)

## 4. Data Analysis & Insights

Seven business questions are answered with Pandas. Each answer ends with an insight written from the real numbers.

### Q1. Which product categories generate the most revenue?

In [ ]:
cat_rev = (sales.groupby("category_name_en")
           .agg(revenue=("price", "sum"), orders=("order_id", "nunique"), items_sold=("order_item_id", "count"))
           .sort_values("revenue", ascending=False))
cat_rev["share_pct"] = (100 * cat_rev["revenue"] / cat_rev["revenue"].sum()).round(2)
top10 = cat_rev.head(10)
display(top10.round(2))

fig, ax = plt.subplots(figsize=(9, 5))
ordered = top10.iloc[::-1]
labels = [textwrap.shorten(c.replace("_", " "), 30, placeholder="...") for c in ordered.index]
ax.barh(labels, ordered["revenue"], color=BLUE, height=0.6)
ax.xaxis.set_major_formatter(THOUSANDS)
ax.set_title("Top 10 product categories by revenue")
ax.set_xlabel("Revenue (R$)")
ax.grid(axis="y", visible=False)
for y, value in enumerate(ordered["revenue"]):
    ax.text(value, y, f"  {value / 1000:,.0f}k", va="center", fontsize=9, color=INK)
save_fig("04_top_categories_revenue.png")

top5_share = cat_rev["share_pct"].head(5).sum()
INSIGHTS.append(
    f"'{cat_rev.index[0]}' is the top category: R$ {cat_rev['revenue'].iloc[0]:,.0f} of revenue, "
    f"{cat_rev['share_pct'].iloc[0]}% of all sales. The 5 biggest categories together make up "
    f"{top5_share:.1f}% of revenue, "
    + ("so sales depend on a few categories." if top5_share >= 40 else "so sales are spread across many categories."))

### Q2. Which month has the highest revenue, and how do sales change over time?

In [ ]:
monthly = (sales.groupby("year_month")
           .agg(revenue=("price", "sum"), orders=("order_id", "nunique")).reset_index())
display(monthly.sort_values("revenue", ascending=False).head(5).round(2))

fig, ax = plt.subplots()
x = pd.to_datetime(monthly["year_month"])
ax.plot(x, monthly["revenue"], color=BLUE, linewidth=2)
ax.set_title("Monthly revenue")
ax.yaxis.set_major_formatter(THOUSANDS)
ax.set_ylabel("Revenue (R$)")
best = monthly.loc[monthly["revenue"].idxmax()]
ax.scatter([pd.to_datetime(best["year_month"])], [best["revenue"]], color=BLUE, s=40, zorder=3)
ax.annotate(f"Highest: {best['year_month']}  (R$ {best['revenue']:,.0f})",
            (pd.to_datetime(best["year_month"]), best["revenue"]), xytext=(-150, -6),
            textcoords="offset points", fontsize=9, color=INK)
save_fig("05_monthly_revenue.png")

INSIGHTS.append(
    f"{best['year_month']} had the highest revenue (R$ {best['revenue']:,.0f}), "
    f"{best['revenue'] / monthly['revenue'].mean():.1f} times the average month "
    f"(R$ {monthly['revenue'].mean():,.0f}).")

### Q3. Which states have the most customers?

In [ ]:
state_customers = (customers.groupby("state")["customer_unique_id"].nunique()
                   .sort_values(ascending=False))
state_share = (100 * state_customers / state_customers.sum()).round(1)
display(pd.DataFrame({"customers": state_customers, "share_pct": state_share}).head(10))

top_states = state_customers.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(top_states.index, top_states.values, color=BLUE, height=0.6)
ax.set_title("Top 10 states by number of customers")
ax.set_xlabel("Unique customers")
ax.grid(axis="y", visible=False)
for y, value in enumerate(top_states.values):
    ax.text(value, y, f"  {value:,}", va="center", fontsize=9, color=INK)
save_fig("06_customers_by_state.png")

INSIGHTS.append(
    f"{state_customers.index[0]} has the most customers ({state_customers.iloc[0]:,}, {state_share.iloc[0]}% of all). "
    f"The top 3 states ({', '.join(state_customers.index[:3])}) hold {state_share.head(3).sum():.1f}% of customers.")

### Q4. How do customers pay?

In [ ]:
pay = (payments.groupby("payment_type")
       .agg(total_value=("payment_value", "sum"), orders=("order_id", "nunique"),
            avg_installments=("payment_installments", "mean"))
       .sort_values("total_value", ascending=False))
pay["share_pct"] = (100 * pay["total_value"] / pay["total_value"].sum()).round(1)
display(pay.round(2))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(pay.index.str.replace("_", " "), pay["total_value"], color=BLUE, width=0.55)
ax.yaxis.set_major_formatter(THOUSANDS)
ax.set_title("Payment value by payment type")
ax.set_ylabel("Total paid (R$)")
ax.grid(axis="x", visible=False)
for xpos, (value, share) in enumerate(zip(pay["total_value"], pay["share_pct"])):
    ax.text(xpos, value, f"{share}%", ha="center", va="bottom", fontsize=9, color=INK)
save_fig("07_payment_types.png")

INSIGHTS.append(
    f"'{pay.index[0]}' is the most used payment method: {pay['share_pct'].iloc[0]}% of all money paid, "
    f"with {pay['avg_installments'].iloc[0]:.1f} installments on average.")

### Q5. What is the average order value?

In [ ]:
aov = order_totals["order_total"].mean()
median_order = order_totals["order_total"].median()
print(f"Orders: {len(order_totals):,}")
print(f"Average order value (items + freight): R$ {aov:,.2f}")
print(f"Median order value: R$ {median_order:,.2f}")

by_state_aov = (order_totals.groupby("state")["order_total"].agg(["mean", "count"])
                .query("count >= 100").sort_values("mean", ascending=False).round(2))
display(by_state_aov.head(5))

if aov > 1.1 * median_order:
    aov_note = "a few expensive orders pull the average up, so the typical customer spends less than the average suggests."
else:
    aov_note = "the two are close, so order values are fairly even."
INSIGHTS.append(
    f"The average order value is R$ {aov:,.2f} and the median is R$ {median_order:,.2f}: " + aov_note)

### Q6. Do late deliveries get worse reviews?

In [ ]:
delivered = (orders[orders["order_status"] == "delivered"]
             .merge(reviews[["order_id", "review_score"]], on="order_id"))
delivered = delivered.dropna(subset=["is_late", "delivery_days"])

late_pct = 100 * delivered["is_late"].mean()
score_late = delivered.loc[delivered["is_late"] == 1, "review_score"].mean()
score_ok = delivered.loc[delivered["is_late"] == 0, "review_score"].mean()
print(f"Late deliveries: {late_pct:.1f}% | average review: late {score_late:.2f} vs on time {score_ok:.2f}")

delivered["delivery_bucket"] = pd.cut(delivered["delivery_days"], [0, 7, 14, 21, 30, np.inf],
                                      labels=["0-7", "8-14", "15-21", "22-30", "over 30"])
by_bucket = delivered.groupby("delivery_bucket", observed=True)["review_score"].agg(["mean", "count"]).round(2)
display(by_bucket)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(by_bucket.index.astype(str), by_bucket["mean"], color=BLUE, width=0.55)
ax.set_title("Average review score by delivery time")
ax.set_xlabel("Days from purchase to delivery")
ax.set_ylabel("Average review score (1-5)")
ax.set_ylim(0, 5)
ax.grid(axis="x", visible=False)
for xpos, value in enumerate(by_bucket["mean"]):
    ax.text(xpos, value, f"{value:.2f}", ha="center", va="bottom", fontsize=9, color=INK)
save_fig("08_delivery_vs_review.png")

delay_verdict = ("slow delivery clearly hurts customer satisfaction." if score_late < score_ok
                 else "late delivery does not lower the average score in this data.")
INSIGHTS.append(
    f"{late_pct:.1f}% of delivered orders arrived after the estimated date. Late orders score "
    f"{score_late:.2f} on average, against {score_ok:.2f} for on-time orders: " + delay_verdict)
INSIGHTS.append(
    f"Average review score is {by_bucket['mean'].iloc[0]:.2f} when delivery takes up to 7 days and "
    f"{by_bucket['mean'].iloc[-1]:.2f} when it takes over 30 days.")

### Q7. How many customers buy again?

In [ ]:
segments = customer_orders["segment"].value_counts().reindex(["One-time", "Two orders", "3+ orders"])
repeat_pct = 100 * (customer_orders["orders"] > 1).mean()
repeat_revenue_pct = 100 * customer_orders.loc[customer_orders["orders"] > 1, "total_spent"].sum() / customer_orders["total_spent"].sum()
display(segments.to_frame("customers"))
top_spenders = customer_orders.sort_values("total_spent", ascending=False).head(5).round(2)
display(top_spenders)

repeat_note = ("Most customers buy a single time, so repeat-purchase campaigns are a big growth opportunity."
               if repeat_pct < 50 else "Most customers come back, which shows strong loyalty.")
INSIGHTS.append(
    f"{repeat_pct:.1f}% of customers ordered more than once; they bring {repeat_revenue_pct:.1f}% of revenue. " + repeat_note)

print("\nKEY INSIGHTS")
for i, insight in enumerate(INSIGHTS, 1):
    print(f"{i}. {insight}")
STATS["insights"] = list(INSIGHTS)

## 5. Database Design

The data is split into **9 related tables**. Full details (primary keys, foreign keys, CHECK constraints, indexes) are in `sql/schema.sql`. The ERD is in `docs/ERD.png`.

| Table | Primary key | Foreign keys | Meaning |
|---|---|---|---|
| `customers` | customer_id | none | one row per order-customer id |
| `sellers` | seller_id | none | sellers |
| `geolocation` | zip_code_prefix | none | one coordinate per zip prefix (lookup) |
| `product_categories` | category_name | none | Portuguese to English names |
| `products` | product_id | category_name → product_categories | products |
| `orders` | order_id | customer_id → customers | one row per order |
| `order_items` | (order_id, order_item_id) | order_id → orders, product_id → products, seller_id → sellers | items inside orders |
| `order_payments` | (order_id, payment_sequential) | order_id → orders | payments of an order |
| `order_reviews` | (review_id, order_id) | order_id → orders | customer reviews |

Relationships: a customer has many orders, an order has many items, payments and reviews, a seller sells many items, and a category has many products.

## 6. Load: Pandas → SQLite

The database is **rebuilt from scratch on every run**: the old file is deleted, the tables are created from `sql/schema.sql`, and the DataFrames are loaded in parent-before-child order. Foreign keys are switched on, so a bad row is rejected by the database itself.

In [ ]:
def db_ready(frame, ts_cols=(), int_cols=()):
    """Turn a DataFrame into plain Python values SQLite accepts: text timestamps, None for missing."""
    out = frame.copy()
    for col in ts_cols:
        out[col] = out[col].dt.strftime("%Y-%m-%d %H:%M:%S")
    for col in int_cols:
        out[col] = out[col].astype("Int64")
    out = out.astype(object)
    return out.where(out.notna(), None)


LOAD_PLAN = [   # parent tables first, child tables after
    ("geolocation",        db_ready(geolocation[["zip_code_prefix", "latitude", "longitude", "city", "state"]])),
    ("customers",          db_ready(customers[["customer_id", "customer_unique_id", "zip_code_prefix", "city", "state"]])),
    ("sellers",            db_ready(sellers[["seller_id", "zip_code_prefix", "city", "state"]])),
    ("product_categories", db_ready(categories[["category_name", "category_name_en"]])),
    ("products",           db_ready(products[["product_id", "category_name", "name_length", "description_length",
                                              "photos_qty", "weight_g", "length_cm", "height_cm", "width_cm"]])),
    ("orders",             db_ready(orders[["order_id", "customer_id", "order_status", "purchase_ts", "approved_ts",
                                            "delivered_carrier_ts", "delivered_customer_ts", "estimated_delivery_date",
                                            "purchase_year", "purchase_month", "delivery_days", "delay_days", "is_late"]],
                                    ts_cols=["purchase_ts", "approved_ts", "delivered_carrier_ts",
                                             "delivered_customer_ts", "estimated_delivery_date"],
                                    int_cols=["is_late"])),
    ("order_items",        db_ready(items[["order_id", "order_item_id", "product_id", "seller_id",
                                           "shipping_limit_ts", "price", "freight_value", "total_price"]],
                                    ts_cols=["shipping_limit_ts"])),
    ("order_payments",     db_ready(payments[["order_id", "payment_sequential", "payment_type",
                                             "payment_installments", "payment_value"]])),
    ("order_reviews",      db_ready(reviews[["review_id", "order_id", "review_score", "comment_title", "comment_message",
                                             "creation_date", "answer_ts"]],
                                    ts_cols=["creation_date", "answer_ts"])),
]
print("Prepared", len(LOAD_PLAN), "tables for loading.")

In [ ]:
# Database connection (SQLAlchemy) with foreign keys switched on for every connection
DB_PATH.unlink(missing_ok=True)                                 # start fresh: the load is repeatable
engine = create_engine(f"sqlite:///{DB_PATH}")

@event.listens_for(engine, "connect")
def _enable_foreign_keys(dbapi_connection, _record):
    dbapi_connection.execute("PRAGMA foreign_keys = ON")

# Create the tables from the SQL file
schema_sql = (SQL_DIR / "schema.sql").read_text(encoding="utf-8")
with sqlite3.connect(DB_PATH) as con:
    con.executescript(schema_sql)
print("Tables created from sql/schema.sql")

# Load: DataFrame -> table
loaded = {}
for table, frame in LOAD_PLAN:
    frame.to_sql(table, engine, if_exists="append", index=False, chunksize=5000)
    loaded[table] = len(frame)
    print(f"  loaded {table:20} {len(frame):>9,} rows")
STATS["loaded_rows"] = loaded

In [ ]:
# Verify the load: row counts, foreign keys, database integrity
with engine.connect() as con:
    in_db = {t: con.execute(text(f"SELECT COUNT(*) FROM {t}")).scalar() for t, _ in LOAD_PLAN}
    fk_problems = con.execute(text("PRAGMA foreign_key_check")).fetchall()
    integrity = con.execute(text("PRAGMA integrity_check")).scalar()

verify = pd.DataFrame({"dataframe_rows": loaded, "database_rows": in_db})
verify["match"] = verify["dataframe_rows"] == verify["database_rows"]
display(verify)
assert verify["match"].all(), "Row counts do not match after loading"
assert not fk_problems, f"Foreign key problems: {fk_problems[:5]}"
assert integrity == "ok", integrity
print(f"Load verified: all row counts match, 0 foreign key problems, integrity check = '{integrity}'.")
STATS["db_size_mb"] = round(DB_PATH.stat().st_size / 1e6, 1)

## 7. SQL Analysis & Reporting

The 11 queries in `sql/reports.sql` run against the database. They use `SELECT`, `WHERE`, `GROUP BY`, `HAVING`, `ORDER BY`, aggregate functions, `JOIN`, `CASE`, subqueries, CTEs and window functions (`RANK`, `LAG`, `SUM() OVER`). Each result is saved as a CSV file in `outputs/sql_reports/`.

In [ ]:
def parse_reports(path):
    """Read reports.sql and return a list of (name, question, sql)."""
    text_ = path.read_text(encoding="utf-8")
    parts = re.split(r"^-- name:\s*(\S+)\s*$", text_, flags=re.M)
    queries = []
    for i in range(1, len(parts), 2):
        name, body = parts[i], parts[i + 1]
        question = re.search(r"^-- question:\s*(.+)$", body, flags=re.M)
        sql = "\n".join(line for line in body.splitlines() if not line.startswith("-- question:")).strip().rstrip(";")
        queries.append((name, question.group(1).strip() if question else "", sql))
    return queries


QUERIES = parse_reports(SQL_DIR / "reports.sql")
SQL_RESULTS = {}
for name, question, sql in QUERIES:
    with engine.connect() as con:
        result = pd.read_sql_query(sql, con)
    result.to_csv(OUT_DIR / f"{name}.csv", index=False)
    SQL_RESULTS[name] = (question, sql, result)
    print(f"\n### {name}\n{question}")
    display(result.head(10))
print(f"\n{len(QUERIES)} reports ran and were saved in {OUT_DIR}")
assert len(QUERIES) >= 8, "At least 8 SQL reports are required"
STATS["sql_report_count"] = len(QUERIES)

**Cross-check:** the SQL answers should match the Pandas answers. For example, the top category in query 1 should be the top category from Q1 in section 4, and the average order value from query 5 should match Q5. A difference would point to a mistake in the cleaning or the SQL.

In [ ]:
sql_top_category = SQL_RESULTS["01_revenue_by_category"][2].iloc[0]["category"]
sql_aov = float(SQL_RESULTS["05_average_order_value"][2].iloc[0]["avg_order_value"])
print(f"Top category   - Pandas: {cat_rev.index[0]!r}   SQL: {sql_top_category!r}")
print(f"Average order  - Pandas: {aov:,.2f}   SQL: {sql_aov:,.2f}")
assert sql_top_category == cat_rev.index[0], "Pandas and SQL disagree on the top category"
assert abs(sql_aov - round(aov, 2)) < 0.01, "Pandas and SQL disagree on average order value"
print("Pandas and SQL agree.")

## 8. Write the project files (combined SQL file, results, report)

This last step creates: the single `.sql` deliverable (`sql/olist_ecommerce.sql` = tables + reporting queries), `docs/sql_report_results.md` (every query with its output) and `docs/Project_Report.md` (the written report, filled with the numbers from this run).

In [ ]:
def md_table(frame, max_rows=15):
    shown = frame.head(max_rows)
    cols = [str(c) for c in shown.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, row in shown.iterrows():
        cells = []
        for v in row:
            cells.append(f"{v:,.2f}" if isinstance(v, float) else str(v))
        lines.append("| " + " | ".join(cells) + " |")
    if len(frame) > max_rows:
        lines.append(f"\n*(first {max_rows} of {len(frame)} rows)*")
    return "\n".join(lines)


# 1) One combined .sql file: table creation + reporting queries
reports_sql = (SQL_DIR / "reports.sql").read_text(encoding="utf-8")
(SQL_DIR / "olist_ecommerce.sql").write_text(
    "-- ===== PART 1: DATABASE AND TABLE CREATION =====\n" + schema_sql
    + "\n\n-- ===== PART 2: SQL REPORTING QUERIES =====\n" + reports_sql, encoding="utf-8")

# 2) Every query with its result
parts = ["# SQL Reports: queries and results\n",
         f"Database: `database/olist.db` (SQLite). {len(SQL_RESULTS)} reporting queries.\n"]
for name, (question, sql, result) in SQL_RESULTS.items():
    parts += [f"\n## {name}\n", f"**Question:** {question}\n", "```sql\n" + sql + ";\n```\n",
              "**Result:**\n", md_table(result), ""]
(ROOT / "docs" / "sql_report_results.md").write_text("\n".join(parts), encoding="utf-8")
print("Wrote sql/olist_ecommerce.sql and docs/sql_report_results.md")

In [ ]:
# 3) The written project report, filled with this run's numbers
shapes = STATS["raw_shapes"]
total_rows = sum(r for r, _ in shapes.values())
clean = STATS["clean_shapes"]
n_orders = shapes["orders"][0]
first_ts = purchase_ts_raw.min().date()
last_ts = purchase_ts_raw.max().date()

# Pieces of the report are built first, so the big text below stays simple
extract_table = md_table(pd.DataFrame({"file": list(shapes.keys()),
                                       "rows": [v[0] for v in shapes.values()],
                                       "columns": [v[1] for v in shapes.values()]}), 20)
nulls_table = md_table(STATS["null_report"].head(8), 8)
log_df = pd.DataFrame(CLEAN_LOG, columns=["Table", "What was done", "Rows affected", "Why"])
cleaning_table = md_table(log_df[log_df["Rows affected"] > 0], 60)

uc, kd, inv = STATS["unique_counts"], STATS["key_dups"], STATS["invalid"]
peak_month, peak_orders = STATS["eda_peak_orders_month"]
loaded_line = ", ".join(f"{k} {v:,}" for k, v in STATS["loaded_rows"].items())
insight_lines = "\n".join(f"{i}. {t}" for i, t in enumerate(STATS["insights"], 1))
sql_lines = "\n".join(f"{i}. **{n}**: {q}" for i, (n, (q, _s, _r)) in enumerate(SQL_RESULTS.items(), 1))
n_not_defined = inv["payments: payment_type = 'not_defined'"]
n_zero_installments = inv["payments: payment_installments = 0"]
n_zero_weight = inv["products: weight_g = 0"]
n_dup_reviews = kd["reviews: repeated (review_id, order_id)"]
n_dup_review_ids = kd["reviews: repeated review_id"]
n_geo_dups = STATS["full_row_dups"]["geolocation"]
satisfaction_note = "most customers are satisfied" if STATS["eda_five_star_pct"] >= 50 else "many customers are not fully satisfied"

report = f"""# Project Report: End-to-End ETL & Data Analysis Pipeline

**Student:** Bipana Khadka
**Dataset:** Brazilian E-Commerce Public Dataset by Olist (Kaggle)
**Tools:** Python, Pandas, Matplotlib, SQLAlchemy, SQLite, SQL

## 1. Dataset and Source
The data comes from the public Kaggle dataset *Brazilian E-Commerce Public Dataset by Olist*
(https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce). It describes orders placed on a Brazilian online
marketplace between {first_ts} and {last_ts}. It has **{len(shapes)} CSV files, {STATS["total_columns"]} columns and {total_rows:,} rows** in total,
including **{n_orders:,} orders**. The files cover orders, order items, customers, products, sellers, payments, reviews,
geolocation and a category translation table. Money is in Brazilian reais (R$).

## 2. Objective
Build a complete and repeatable ETL pipeline: extract the raw files, explore and clean them with Python and Pandas,
transform them into related tables, load them into a relational SQLite database, and use SQL and Pandas to find business
insights about sales, customers, payments and delivery.

## 3. Extraction
All {len(shapes)} CSV files are read with `pandas.read_csv` into DataFrames. The original files in `data/raw/` are never edited;
all cleaning is done on copies.

{extract_table}

## 4. EDA Findings
- The busiest month in the raw data was **{peak_month}** with {peak_orders:,} orders (see `docs/figures/01_orders_per_month.png`).
- {STATS["eda_five_star_pct"]}% of all reviews give 5 stars, so {satisfaction_note}.
- The median item price is R$ {STATS["eda_price_median"]}, with a long tail of expensive items.
- `customer_id` is created for every order, while `customer_unique_id` identifies the real person: there are {uc["customers.customer_id"]:,} customer ids but only {uc["customers.customer_unique_id"]:,} real customers.
- Customers come from {uc["customers.customer_state"]} states, and {uc["products.product_category_name"]} product categories exist.

## 5. Data Quality Issues Found
**Missing values (largest):**

{nulls_table}

**Other issues:**
- {STATS["wrong_types"]} date columns were stored as plain text.
- Duplicates: the reviews file has {n_dup_reviews:,} repeated (review_id, order_id) pairs, and the geolocation file has {n_geo_dups:,} fully duplicated rows.
- Invalid values: {n_not_defined} payments with type not_defined, {n_zero_installments} payments with 0 installments, {n_zero_weight} products with weight 0 g.
- {len(STATS["untranslated"])} product categories had no English translation.
- City names had accents, extra spaces and mixed capital letters.
- Outliers: very expensive items exist, but they are real purchases, so they were kept.

## 6. Cleaning and Transformation
Every step is based on a problem found during EDA:

{cleaning_table}

New analysis tables: `sales` (one row per sold item), `order_totals` (one row per order) and `customer_orders` (one row per real customer with a segment).
{STATS["quality_checks"]} automatic quality checks must pass before anything is loaded.

## 7. Database Design
The database has **9 related tables**: customers, sellers, geolocation, product_categories, products, orders, order_items, order_payments, order_reviews.
Each table has a primary key, and foreign keys link them (customer, then orders, then items / payments / reviews; products and sellers, then items;
category, then products). CHECK constraints protect the data (for example price at least 0, review score 1 to 5, valid order status and payment type).
The ERD is in `docs/ERD.png` and the SQL is in `sql/schema.sql`.

![ERD](ERD.png)

## 8. ETL Process
1. **Extract:** read the 9 CSV files with Pandas.
2. **Transform:** explore, check quality, clean and build new columns with Pandas.
3. **Load:** delete the old database, create the tables from `schema.sql`, and load the DataFrames with SQLAlchemy (`to_sql`) in parent-before-child order.
4. **Verify:** row counts, foreign key check and integrity check.
5. **Report:** run the SQL queries and save the results.

Loaded rows: {loaded_line}.
The whole process is repeatable with `python run_pipeline.py` (or by running the notebook).

## 9. Data Analysis
Seven questions were answered with Pandas: top categories by revenue, the best month, customers by state, payment methods,
average order value, delivery time versus reviews, and repeat customers.

![Top categories](figures/04_top_categories_revenue.png)
![Monthly revenue](figures/05_monthly_revenue.png)
![Delivery vs review](figures/08_delivery_vs_review.png)

## 10. Key Insights
{insight_lines}

## 11. SQL Reports
{len(SQL_RESULTS)} queries (see `sql/reports.sql`; results in `docs/sql_report_results.md` and `outputs/sql_reports/`):

{sql_lines}

## 12. Challenges Encountered
- **Repeated review ids:** {n_dup_review_ids:,} review_id values appear more than once, so `review_id` alone cannot be a primary key. A composite key (review_id, order_id) was used.
- **Customer identity:** `customer_id` changes with every order, so customer analysis uses `customer_unique_id`.
- **Huge geolocation file:** it has {shapes["geolocation"][0]:,} rows with many repeated zip codes. It was reduced to {clean["geolocation"]:,} rows (one per zip prefix). A foreign key from customers to it was not used because some customer zip codes are not in the file.
- **Missing translations and categories:** some products had no category and some categories had no English name; these were handled with an 'unknown' category and a readable fallback name.
- **Dates as text:** all date columns had to be converted before delivery time and monthly trends could be calculated.
- **Keeping the load repeatable:** the database is deleted and rebuilt on each run, with foreign keys switched on so mistakes are caught immediately.

## 13. Conclusion
The project turns {total_rows:,} rows of raw marketplace data across {len(shapes)} files into a clean, validated, relational database and a set of
reports. It shows the complete path: Raw Data, Extract, Explore, Clean, Transform, Analyze, Load, SQL Reports. The business findings
(see Key Insights) show where revenue comes from, how customers pay, and how delivery speed affects satisfaction.
Possible next steps: scheduling the pipeline (for example with Airflow), loading into PostgreSQL, and building a dashboard.
"""
(ROOT / "docs" / "Project_Report.md").write_text(report, encoding="utf-8")
print("Wrote docs/Project_Report.md")
print("\nPIPELINE COMPLETE: raw data extracted, cleaned, loaded into", DB_PATH.name, "and reported.")